# Successeur de gravité

Le score public précédent, transmis par l'utilisateur, est 1365.913. Ce notebook lit les résultats de `scripts/improve_severity.py train` et les contrôles reproductibles. Le nouveau score Kaggle reste inconnu. Les données ont déjà été explorées ; l'évaluation locale est diagnostique.

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p/'run.py').exists())
R=ROOT/'reports/severity_successor'
r=json.loads((R/'results.json').read_text())
pd.DataFrame({name:r[name] for name in ['selected','incumbent','original']})

,selected,incumbent,original
MSE,293788.504166,293797.307272,293829.184497
RMSE,542.022605,542.030725,542.060130
MAE,191.238886,191.035702,190.861365
bias,0.125736,-0.094337,-0.329742
predicted_observed_ratio,1.001216,0.999087,0.996810


93 choix prédéfinis : 21 corrections, deux moyennes, quatre poids et Gamma seule. Trois plis externes séparés par client évaluent la sélection sur cinq plis internes. Probabilités originales inchangées, corrections entraînées seulement sur les sinistres, pondérations auxiliaires calculées hors client, sinistres extrêmes conservés.

In [2]:
pd.DataFrame([{'fold':x['fold'],'selected_RMSE':x['selected']['RMSE'],'incumbent_RMSE':x['incumbent']['RMSE'],'choice':x['choice']} for x in r['folds']])

,fold,selected_RMSE,incumbent_RMSE,choice
0,1,569.530275,569.519671,"{'candidate': 17, 'weight': 0.25, 'selection_R..."
1,2,518.101342,518.123563,"{'candidate': 17, 'weight': 0.5, 'selection_RM..."
2,3,537.188481,537.202874,"{'candidate': 7, 'weight': 1.0, 'selection_RMS..."


In [3]:
r['gain_vs_incumbent']

{'gain_RMSE': 0.008120546762484082,
 'interval_95': [-0.018781685311986963, 0.03709055955707185],
 'repetitions': 2000,
 'interpretation': 'Descriptive paired client bootstrap conditional on fitted predictions; not full training uncertainty.'}

L'intervalle bootstrap inclut zéro : le gain n'est pas établi avec précision. La recette fixe du modèle à 1365.913 est réapprise par pli. Cette comparaison diffère de l'ancienne recherche qui sélectionnait un modèle de gravité par pli.

In [4]:
selection=json.loads((R/'full_training/selection.json').read_text())
config=selection['catalogue'][r['final_choice']['candidate']]
print('Poids de correction :',r['final_choice']['weight'])
config

Poids de correction : 0.5


{'name': 'bagged_480',
 'panel': 'basic',
 'mode': 'ratio',
 'depth': 2,
 'trees': 480,
 'lr': 0.025,
 'min_child_weight': 100.0,
 'reg_lambda': 100.0,
 'reg_alpha': 1.0,
 'max_bin': 256,
 'seeds': 3,
 'subsample': 0.8}

In [5]:
verification=json.loads((R/'verification.json').read_text())
assert verification['status']=='verified'
assert verification['frequency_unchanged'] and verification['fresh_process_byte_identical']
assert verification['full_refit_incumbent_exact']
verification

{'status': 'verified',
 'tests_passed': 43,
 'rows': 50000,
 'fresh_process_byte_identical': True,
 'frequency_unchanged': True,
 'full_refit_incumbent_exact': True,
 'incumbent_csv_unchanged': True,
 'original_notebook_sha256': '76248bdb51400fe399c670a9b71092cabe041cc13d3839b83846ac1499938429',
 'candidate_sha256': '28ab50d597186e0e5c3a06cb4b354ee31feba09c3b3c1ff83179b9e16d843e89',
 'model_sha256': 'ef262fe49b6ca58f55b032184dcf377d83aa07713b717386939a6fc7c359edba',
 'script_sha256': '4d1971560653dce0e765bc23c61b0c03667de6fbda57e24fe32e6d019efa287a',
 'provenance': {'data': {'train.csv': '8b5ca8a59c214269056ed1720de33619c10968549c383ec350898a659b97bfc5',
   'test.csv': '68cdf87201974688d754d846c2d8ff33c5aa2f9c011e84fba92018186cd8f569'},
  'source': {'__init__.py': '100b3e376bb5c6d141565df62567e20d231861a23f17cb8e15d587cce02ee61e',
   'experiment.py': '0237a7515b456a562c85c8b5e7339f2d08ca5a179c75bb7c7da33f6c86522eab',
   'models.py': 'a2787421cff97fa7ca2f7fb21d863eec035c61cd6826f459bbcb

In [6]:
pd.read_csv(ROOT/r['export']['file']).describe()

,index,pred
count,50000.000000,50000.000000
mean,74999.500000,102.644950
std,14433.901067,61.503762
min,50000.000000,1.027003
25%,62499.750000,48.069199
50%,74999.500000,101.461819
75%,87499.250000,144.834884
max,99999.000000,544.787720


Export : `deliverables/candidates/submission_severity_successor.csv`, 50 000 lignes index,pred. L'ancien meilleur CSV et le notebook original sont préservés. Aucun score Kaggle nouveau n'est revendiqué. Commandes et détails : `docs/severity_successor.md`.